In [ ]:
# How `F.linear` works

Expert habit: **print shapes → compute two ways → assert they match**.

Router context: `F.linear(x, W)` with `W` shaped `(n_experts, hidden)` scores every token against every expert.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

# one token, hidden=4; weight: 3 experts × 4 hidden
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])          # (1, 4)
W = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],   # expert 0: pick x0
    [0.0, 1.0, 0.0, 0.0],   # expert 1: pick x1
    [0.25, 0.25, 0.25, 0.25],  # expert 2: average
])                                            # (3, 4)

y_linear = F.linear(x, W)                     # (1, 3)
y_matmul = x @ W.T                            # same math: y = x W^T

print("x       ", tuple(x.shape), x)
print("W       ", tuple(W.shape))
print("F.linear", tuple(y_linear.shape), y_linear)
print("x @ W.T ", tuple(y_matmul.shape), y_matmul)
assert torch.allclose(y_linear, y_matmul)
print("PASS: F.linear == x @ W.T")

## Same op on a batch of tokens (MoE router shape)

Flatten `(B, S, H) → (B·S, H)`, then linear → `(B·S, n_experts)`.

In [ ]:
B, S, H, N = 2, 4, 8, 6   # batch, seq, hidden, n_experts

hidden = torch.randn(B, S, H)
W_gate = torch.randn(N, H)                 # like TopkRouter.weight

flat = hidden.view(-1, H)                  # (8, 8)
logits = F.linear(flat, W_gate)            # (8, 6)

print("hidden ", tuple(hidden.shape))
print("flat   ", tuple(flat.shape))
print("W_gate ", tuple(W_gate.shape))
print("logits ", tuple(logits.shape), "← one score per expert, per token")
print("token0 scores:", logits[0])

# linear also works without flattening (applies on last dim)
logits_3d = F.linear(hidden, W_gate)       # (2, 4, 6)
assert torch.allclose(logits, logits_3d.view(-1, N))
print("PASS: flat linear matches 3D linear")

## `nn.Linear` vs `F.linear`

Module stores the weight; functional takes the weight as an argument. Same math.

In [ ]:
layer = torch.nn.Linear(H, N, bias=False)
layer.weight.data.copy_(W_gate)            # same W as above

y_mod = layer(flat)
y_fn  = F.linear(flat, layer.weight)

assert torch.allclose(y_mod, y_fn)
print("PASS: nn.Linear(x) == F.linear(x, layer.weight)")
print(y_mod.shape)